# PARSER-001 - Text preprocessing dan amount parser

Notebook ini berisi prototype parser nominal bahasa Indonesia informal sesuai PRD Personal Finance Assistant Bot.

Target issue:
- Normalisasi lowercase.
- Baca format `20 ribu`, `20rb`, `20k`.
- Baca format `Rp20.000`.
- Baca format `2 juta`, `2jt`.
- Konversi nominal ke integer.
- Jika nominal tidak ada, return `need_clarification`.

Notebook ini dibuat sebagai langkah awal sebelum logic dikonversi ke modul Python production.

In [ ]:
from dataclasses import dataclass
from decimal import Decimal, InvalidOperation, ROUND_HALF_UP
import re
from typing import Iterable


In [ ]:
AMOUNT_PATTERN = re.compile(
    r"(?<![\w])"
    r"(?P<prefix>rp\.?\s*)?"
    r"(?P<number>\d+(?:[.,]\d{3})*(?:[.,]\d+)?)"
    r"\s*(?P<unit>juta|jt|ribu|rb|k)?"
    r"\b",
    re.IGNORECASE,
)

UNIT_MULTIPLIERS = {
    "ribu": Decimal("1000"),
    "rb": Decimal("1000"),
    "k": Decimal("1000"),
    "juta": Decimal("1000000"),
    "jt": Decimal("1000000"),
}

MIN_UNQUALIFIED_AMOUNT = 1000


@dataclass(frozen=True)
class AmountCandidate:
    amount: int
    matched_text: str
    start: int
    end: int
    score: float


@dataclass(frozen=True)
class AmountParseResult:
    raw_text: str
    normalized_text: str
    amount: int | None
    matched_text: str | None
    status: str
    need_clarification: bool
    reason: str | None = None


## Helper preprocessing

`normalize_text` hanya melakukan lowercase dan merapikan whitespace. Normalisasi lain sengaja minimal agar input asli user tetap mudah dipakai untuk deskripsi transaksi.

In [ ]:
def normalize_text(text: str) -> str:
    return re.sub(r"\s+", " ", text.strip().lower())


def parse_amount(text: str) -> AmountParseResult:
    normalized = normalize_text(text)
    candidates = list(_iter_amount_candidates(normalized))

    if not candidates:
        return AmountParseResult(
            raw_text=text,
            normalized_text=normalized,
            amount=None,
            matched_text=None,
            status="need_clarification",
            need_clarification=True,
            reason="missing_amount",
        )

    best = max(candidates, key=lambda candidate: candidate.score)
    return AmountParseResult(
        raw_text=text,
        normalized_text=normalized,
        amount=best.amount,
        matched_text=best.matched_text,
        status="parsed",
        need_clarification=False,
    )


def _iter_amount_candidates(text: str) -> Iterable[AmountCandidate]:
    for match in AMOUNT_PATTERN.finditer(text):
        prefix = match.group("prefix")
        number = match.group("number")
        unit = match.group("unit")
        amount = _parse_amount_value(
            number,
            unit,
            has_rupiah_prefix=bool(prefix),
        )

        if amount is None or amount <= 0:
            continue

        if amount < MIN_UNQUALIFIED_AMOUNT and not prefix and not unit:
            continue

        yield AmountCandidate(
            amount=amount,
            matched_text=match.group(0),
            start=match.start(),
            end=match.end(),
            score=_score_amount_match(
                amount=amount,
                text_length=len(text),
                start=match.start(),
                has_rupiah_prefix=bool(prefix),
                has_unit=bool(unit),
                number=number,
            ),
        )


def _parse_amount_value(
    number: str,
    unit: str | None,
    *,
    has_rupiah_prefix: bool,
) -> int | None:
    unit_normalized = (unit or "").lower()

    try:
        if unit_normalized:
            value = _parse_decimal_number(number) * UNIT_MULTIPLIERS[unit_normalized]
        elif has_rupiah_prefix or _looks_like_grouped_currency(number):
            value = Decimal(re.sub(r"[^\d]", "", number))
        else:
            value = Decimal(number)
    except (InvalidOperation, KeyError, ValueError):
        return None

    return int(value.quantize(Decimal("1"), rounding=ROUND_HALF_UP))


def _parse_decimal_number(number: str) -> Decimal:
    if "." in number and "," in number:
        if number.rfind(",") > number.rfind("."):
            normalized = number.replace(".", "").replace(",", ".")
        else:
            normalized = number.replace(",", "")
    elif "," in number:
        normalized = number.replace(",", ".")
    elif "." in number:
        parts = number.split(".")
        normalized = "".join(parts) if len(parts[-1]) == 3 else number
    else:
        normalized = number

    return Decimal(normalized)


def _looks_like_grouped_currency(number: str) -> bool:
    return bool(re.fullmatch(r"\d{1,3}([.,]\d{3})+", number))


def _score_amount_match(
    *,
    amount: int,
    text_length: int,
    start: int,
    has_rupiah_prefix: bool,
    has_unit: bool,
    number: str,
) -> float:
    score = 0.0
    if has_rupiah_prefix:
        score += 4.0
    if has_unit:
        score += 3.0
    if _looks_like_grouped_currency(number):
        score += 2.0
    if amount >= MIN_UNQUALIFIED_AMOUNT:
        score += 1.0

    score -= start / max(text_length, 1)
    return score


## Test cases PRD dan variasi informal

Case utama diambil dari PRD: `20 ribu`, `20rb`, `20k`, `Rp20.000`, `2 juta`, serta fallback `need_clarification` jika nominal tidak ada.

In [ ]:
TEST_CASES = [
    ("beli makan 20 ribu", 20_000, "parsed"),
    ("beli makan 20rb", 20_000, "parsed"),
    ("beli makan 20k", 20_000, "parsed"),
    ("beli makan Rp20.000", 20_000, "parsed"),
    ("beli makan rp 20.000", 20_000, "parsed"),
    ("gaji masuk 2 juta", 2_000_000, "parsed"),
    ("bonus 2jt", 2_000_000, "parsed"),
    ("bayar bensin 30rb kemarin", 30_000, "parsed"),
    ("tagihan internet Rp250.000", 250_000, "parsed"),
    ("tabungan 1,5 juta", 1_500_000, "parsed"),
    ("belanja 2 item Rp20.000", 20_000, "parsed"),
    ("beli kopi", None, "need_clarification"),
]


for text, expected_amount, expected_status in TEST_CASES:
    result = parse_amount(text)
    assert result.amount == expected_amount, (text, result.amount, expected_amount)
    assert result.status == expected_status, (text, result.status, expected_status)

print(f"OK - {len(TEST_CASES)} amount parser test cases passed")


In [ ]:
for text, _, _ in TEST_CASES:
    result = parse_amount(text)
    print(
        {
            "input": text,
            "normalized": result.normalized_text,
            "matched": result.matched_text,
            "amount": result.amount,
            "status": result.status,
            "reason": result.reason,
        }
    )


## Catatan konversi ke Python

Saat dikonversi ke production module, fungsi inti yang perlu dipindahkan adalah:
- `normalize_text`
- `parse_amount`
- `_iter_amount_candidates`
- `_parse_amount_value`
- `_parse_decimal_number`
- `_looks_like_grouped_currency`
- `_score_amount_match`

Rekomendasi integrasi:
- Masukkan ke `backend/app/modules/parser/transaction_text.py` atau pisahkan menjadi `backend/app/modules/parser/amount.py`.
- Samakan response `missing_amount` dengan contract production: `need_clarification=True` atau mapping ke `need_confirmation=True` jika masih memakai dataclass lama.
- Tambahkan unit test production untuk semua `TEST_CASES` di notebook ini.